In [8]:
import os
import json
import requests
import msal
from dotenv import load_dotenv

# Securely load variables from the .env file
load_dotenv()

# Configuration
CLIENT_ID = os.getenv("CLIENT_ID")
AUTHORITY = "https://login.microsoftonline.com/consumers"
SCOPES = ["Tasks.ReadWrite", "User.Read"]

if not CLIENT_ID:
    print("Error: CLIENT_ID not found. Please check your .env file.")

In [20]:
def get_access_token():
    app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY)

    # Opens a browser window for you to log in securely
    result = app.acquire_token_interactive(scopes=SCOPES)

    if "access_token" in result:
        print("Login successful! Token acquired.")
        return result["access_token"]
    else:
        print("Error logging in:", result.get("error_description"))
        return None

In [21]:
def load_tasks_from_json(filepath):
    try:
        with open(filepath, "r") as file:
            data = json.load(file)
            print(f"Successfully loaded {len(data)} tasks.")
            # print(data)
            return data
    except FileNotFoundError:
        print(f"Error: Could not find the file '{filepath}'.")
        return []

In [22]:
loaded_tasks = load_tasks_from_json("tasks.json")
print(loaded_tasks)
print(type(loaded_tasks))
print(loaded_tasks[0]["area"])

Successfully loaded 13 tasks.
[{'area': 'Development', 'title': 'Review Python logic', 'notes': 'Scenario 1: Basic task. No dates, no status, no recurrence.'}, {'area': 'Travel', 'title': 'Shop for sister and mother', 'status': 'completed', 'notes': 'Scenario 2: Pre-completed task.'}, {'area': 'Business', 'title': 'Fix Blue Star AC', 'due_date': '2026-07-01', 'importance': 'high', 'notes': 'Scenario 3: Overdue task. Due date is strictly in the past.'}, {'area': 'Business', 'title': 'Edit reels in CapCut', 'due_date': '2026-07-15', 'reminder_time': '2026-07-15T18:00:00', 'notes': 'Scenario 4: Task due today with a specific reminder time.'}, {'area': 'Development', 'title': 'Check Tic-Tac-Toe shuffle code', 'reminder_time': '2026-07-10T10:00:00', 'notes': 'Scenario 5: Reminder set in the past. Tests if Microsoft accepts expired reminders.'}, {'area': 'Personal', 'title': 'Watch Daredevil', 'recurrence': {'type': 'daily', 'interval': 1, 'range_type': 'noEnd', 'start_date': '2026-07-15'}, 

In [23]:
def get_or_create_list(token, list_name):
    # 1. Check for the specific list name directly using $filter
    check_url = f"https://graph.microsoft.com/v1.0/me/todo/lists?$filter=displayName eq '{list_name}'"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    response = requests.get(check_url, headers=headers)

    if response.status_code == 200:
        data = response.json().get("value", [])
        if data:  # If the list exists, the data array will not be empty
            print(f"Found existing list: '{list_name}'")
            return data[0].get("id")

    # 2. If it does not exist, create a new one
    create_url = "https://graph.microsoft.com/v1.0/me/todo/lists"
    body = {"displayName": list_name}
    create_response = requests.post(create_url, headers=headers, json=body)

    if create_response.status_code == 201:
        print(f"Created new list: '{list_name}'")
        return create_response.json().get("id")
    else:
        print("Failed to create list:", create_response.text)
        return None

In [24]:
def upload_detailed_task(token, list_id, task_data):
    url = f"https://graph.microsoft.com/v1.0/me/todo/lists/{list_id}/tasks"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    body = {"title": task_data.get("title", "Untitled Task")}

    if "notes" in task_data:
        body["body"] = {"content": task_data["notes"], "contentType": "text"}
    if "importance" in task_data:
        body["importance"] = task_data["importance"]
    if "status" in task_data:
        body["status"] = task_data["status"]

    # --- THE SAFETY NET ---
    # If it repeats but has no due date, auto-fix it by using the start_date
    if "recurrence" in task_data and "due_date" not in task_data:
        fallback_date = task_data["recurrence"].get("start_date")
        if fallback_date:
            task_data["due_date"] = fallback_date
            print(f"  -> Auto-fixed missing due date for: {task_data.get('title')}")

    # 4. Due Date
    if "due_date" in task_data:
        body["dueDateTime"] = {
            "dateTime": f"{task_data['due_date']}T00:00:00",
            "timeZone": "India Standard Time",
        }

    # 5. Reminders
    if "reminder_time" in task_data:
        body["reminderDateTime"] = {
            "dateTime": f"{task_data['reminder_time']}",
            "timeZone": "India Standard Time",
        }
        body["isReminderOn"] = True

    # 6. Recurrence
    if "recurrence" in task_data:
        rec = task_data["recurrence"]
        pattern = {"type": rec.get("type", "daily"), "interval": rec.get("interval", 1)}

        if "daysOfWeek" in rec:
            pattern["daysOfWeek"] = rec["daysOfWeek"]

        range_data = {
            "type": rec.get("range_type", "noEnd"),
            # Safely get start_date, or fallback to the due_date we just ensured exists
            "startDate": rec.get("start_date", task_data.get("due_date")),
        }

        if "end_date" in rec:
            range_data["endDate"] = rec["end_date"]

        body["recurrence"] = {"pattern": pattern, "range": range_data}

    response = requests.post(url, headers=headers, json=body)

    if response.status_code == 201:
        print(f"  -> Added: {task_data.get('title')}")
    else:
        print(f"  -> Failed to add '{task_data.get('title')}': {response.text}")

In [25]:
# 1. Get the VIP pass
access_token = get_access_token()

if access_token:
    # 2. Read your local file
    my_tasks = load_tasks_from_json("tasks.json")

    # Dictionary to save list IDs so we do not search for the same list twice
    list_cache = {}

    if my_tasks:
        for task in my_tasks:
            area_name = task["area"]

            # 3. Check if we already have the list ID for this area
            if area_name not in list_cache:
                list_cache[area_name] = get_or_create_list(access_token, area_name)

            current_list_id = list_cache.get(area_name)

            if current_list_id:
                # 4. Upload the task with all details
                upload_detailed_task(access_token, current_list_id, task)

        print("All operations completed successfully.")

Login successful! Token acquired.
Successfully loaded 13 tasks.
Found existing list: 'Development'
  -> Added: Review Python logic
Found existing list: 'Travel'
  -> Added: Shop for sister and mother
Found existing list: 'Business'
  -> Added: Fix Blue Star AC
  -> Added: Edit reels in CapCut
  -> Added: Check Tic-Tac-Toe shuffle code
Found existing list: 'Personal'
  -> Auto-fixed missing due date for: Watch Daredevil
  -> Added: Watch Daredevil
  -> Added: Meet accountability partner
  -> Added: Finalize 4-5 day itinerary
  -> Added: Record new content
  -> Added: Find pure veg restaurants
  -> Added: Review web structures
  -> Added: Pay monthly bills
  -> Added: Yearly backup
All operations completed successfully.
